In [1]:
import os
import re
import time
import threading
from datetime import datetime
from flask import Flask, request, jsonify
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
import nest_asyncio

nest_asyncio.apply()
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
NOM_DU_MODELE = "openai/gpt-oss-120b"

app = Flask(__name__)

# ====== RÉGLAGES ======
PROPRIETAIRE = "SATORU GOJO"   # le nom de la personne dont le bot est l'assistant (le nom sous lequel tes contacts te connaissent)
CONTEXT_MESSAGES = 8    # nombre de messages récents donnés au modèle (juste pour comprendre le sujet)

SYSTEM_PROMPT = f"""
Tu es l'assistant de {PROPRIETAIRE}. 
Tu réponds à ses messages WhatsApp pendant qu'il n'est pas disponible. 
Tu n'es PAS {PROPRIETAIRE} : tu ne prétends jamais être lui et tu ne dis jamais "c'est moi".

QUI TU ES :
- Si on te demande qui tu es, ou si on pense parler à {PROPRIETAIRE}, dis la vérité simplement : tu es son assistant et tu réponds à sa place.
- Si on te demande sincèrement si tu es un robot, une IA, un programme ou une vraie personne, réponds honnêtement : tu es un assistant automatisé, pas {PROPRIETAIRE} lui-même.
- Si quelqu'un veut parler à {PROPRIETAIRE} en personne, explique que tu es son assistant et qu'il faudra le contacter directement ou attendre qu'il soit disponible. Ne promets pas de lui transmettre le message.

CE QUE TU NE FAIS PAS :
- Tu n'as pas de vie personnelle : ne parle pas de "ton" travail, "ta" famille, "ton" week-end, et n'invente aucune expérience personnelle.
- N'invente aucune information sur {PROPRIETAIRE} (où il est, ce qu'il fait, ce qu'il pense) et ne donne aucune information privée sur lui.
- Ne prends aucun engagement à sa place (rendez-vous, argent, promesses, décisions).

COMMENT TU ÉCRIS :
- Tu agis comme un assistant humain : naturel, chaleureux, poli, détendu et simple.
- Réponds directement et naturellement au dernier message reçu. Si on te pose une question, réponds à la question.
- Réponds dans la langue du dernier message reçu (Si le message reçu est en Anglais, repond en Anglais. si le message reçu est en francais, repond en francais etc).
- Écris avec de vrais mots : une ou deux phrases courtes. Jamais seulement un emoji, un pouce ou un symbole.
- Pas de réponses sèches comme juste "Ok" ou "Oui".
- Tu peux mettre un emoji de temps en temps, mais jamais seul.
- Pas de listes, pas de titres, pas de gras, pas de markdown. Pas de phrases de robot ("n'hésite pas", "en tant que modèle", "je suis là pour t'aider").
- Le contexte sert uniquement à comprendre de quoi on parle. Ne copie pas le style des anciens messages, surtout pas ceux composés uniquement d'emojis.
- Si tu ne comprends pas le message, demande poliment de préciser.
- Réponds UNIQUEMENT avec le texte du message, sans guillemets ni nom devant.
"""

memory = {}   # secours si aucun historique n'est envoyé

try:
    llm = ChatGroq(
        groq_api_key=groq_api_key,
        model_name=NOM_DU_MODELE,
        temperature=0.7,
        reasoning_effort="low",
        timeout=30,
        max_retries=1
    )
    print("✅ Modèle Groq initialisé avec succès dans Jupyter.")
except Exception as e:
    print(f"⚠️ Erreur d'initialisation Groq : {e}")

def nettoyer(texte):
    texte = re.sub(r'\*\*(.*?)\*\*', r'\1', texte)
    texte = re.sub(r'^#+\s*', '', texte, flags=re.M)
    texte = texte.replace('—', ',')
    texte = re.sub(rf'^\s*(Assistant|{re.escape(PROPRIETAIRE)})\s*:\s*', '', texte)
    return texte.strip()

def a_des_mots(texte):
    # vrai s'il y a au moins une lettre (donc pas uniquement des emojis)
    return bool(re.search(r'[^\W\d_]', texte))

def contexte(history, contact_name):
    autre = contact_name or "Interlocuteur"
    lignes = []
    for m in history[-CONTEXT_MESSAGES:]:
        qui = f"Côté {PROPRIETAIRE}" if m['fromMe'] else autre
        lignes.append(f"{qui} : {m['body']}")
    return "\n".join(lignes)

@app.route("/process-message", methods=['POST'])
def process_message():
    if not request.is_json:
        return jsonify({"error": "Content-Type must be application/json"}), 400

    data = request.get_json()
    sender = data.get('sender', '')
    incoming_msg = data.get('message', '').strip()
    history = data.get('history') or []
    contact_name = data.get('contact_name', '')

    print(f"\n[FLASK - JUPYTER] Message reçu de {sender} : '{incoming_msg}'")

    try:
        if not incoming_msg:
            reply_text = "Hein ? Je n'ai rien reçu"
        else:
            if not history:
                mem = memory.setdefault(sender, [])
                mem.append({'fromMe': False, 'body': incoming_msg})
                history = mem

            prompt = (
                f"Contexte récent de la conversation :\n{contexte(history, contact_name)}\n\n"
                f"Dernier message reçu : {incoming_msg}\n\n"
                f"Écris ta réponse à ce dernier message."
            )

            reply_text = ""
            for essai in range(3):
                t0 = time.time()
                modele = llm if essai == 0 else llm.bind(reasoning_effort="medium")
                ai_response = modele.invoke([SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=prompt)])
                contenu = ai_response.content
                if isinstance(contenu, list):
                    contenu = " ".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in contenu)
                print(f"[FLASK - JUPYTER] Groq a répondu en {time.time() - t0:.1f}s")
                reply_text = nettoyer(contenu or "")
                if reply_text and a_des_mots(reply_text):
                    break
                print(f"⚠️ Réponse vide ou sans mots (essai {essai + 1}) : {reply_text!r}")
                reply_text = ""

            if not reply_text:
                reply_text = "Désolé, je n'ai pas bien compris, tu peux répéter ?"

            if sender in memory:
                memory[sender].append({'fromMe': True, 'body': reply_text})
    except Exception as e:
        print(f"❌ Erreur Groq : {e}")
        reply_text = "Désolé, j'ai eu un petit souci, tu peux répéter ?"

    print(f"[FLASK - JUPYTER] Réponse générée : {reply_text}")
    return jsonify({"reply": reply_text})

def run_flask():
    app.run(host="127.0.0.1", port=5000, use_reloader=False)

flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

print("🚀 Serveur Flask démarré sur http://127.0.0.1:5000")

✅ Modèle Groq initialisé avec succès dans Jupyter.
🚀 Serveur Flask démarré sur http://127.0.0.1:5000
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
